In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [8]:
import warnings

import json

from model.optimization import load_model, eval_model
from model.dataset import get_dataframe
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore")

category="isoforms_min_tpm_3"

In [9]:
test_df = get_dataframe(category=category, dataset="test")
subtypes = test_df["subtype"].unique()

method = "logistic"
metric = "f1"
importances = json.loads(open(f"../../preprocessed/{category}/important_isoforms_{method}_{metric}.json").readline())

In [10]:
def run_tests(isoforms: int, subtype: str, report: bool = True):
  chosen_isoforms = list(set([y["isoform"] for x in [sex_values[:isoforms] for sex_values in importances[subtype].values()] for y in x]))
  print(f"Total chosen isoforms: {len(chosen_isoforms)}")

  train_data = get_dataframe(category=category, dataset="train")
  train_data["subtype_target"] = train_data["subtype"] == subtype

  model = load_model(
    name=f"feature_selection_{method}_{metric}",
    model_factory=lambda **kwargs: HistGradientBoostingClassifier(**kwargs, max_iter=1000, class_weight="balanced"),
    dataset=(train_data[chosen_isoforms], train_data["subtype_target"])
  )
  
  test_data = test_df.copy()
  test_data["subtype_target"] = test_data["subtype"] == subtype
  return eval_model(model, dataset=(test_data[chosen_isoforms], test_data["subtype_target"]), report=report, use_threshold=True)
  

In [28]:
isoform_count_by_subtype = {}
acc_by_subtype = {}
for subtype in subtypes:
  print(f"======== Subtype {subtype} ========")
  
  current_max = 0
  isoforms_max = 0
  for i in range(1, 12):
    acc = run_tests(isoforms=i, subtype=subtype, report=False).balanced_accuracy
    if acc > current_max + 0.01:
      current_max = acc
      isoforms_max = i

  isoform_count_by_subtype[subtype] = isoforms_max
  acc_by_subtype[subtype] = current_max

======== Subtype PHlike ========
Total chosen isoforms: 1
Using threshold 0.2593951397662865, with max acc 0.7252699153778814
Total chosen isoforms: 3
Using threshold 0.24074339736277392, with max acc 0.7437992413189378
Total chosen isoforms: 5
Using threshold 0.42708249187187813, with max acc 0.8042019258826962
Total chosen isoforms: 7
Using threshold 0.3210918218318725, with max acc 0.8309016632623285
Total chosen isoforms: 9
Using threshold 0.34321302402618464, with max acc 0.8376130726583018
Total chosen isoforms: 11
Using threshold 0.28978503621201496, with max acc 0.8376130726583018
Total chosen isoforms: 13
Using threshold 0.21059003548040536, with max acc 0.845929384301138
Total chosen isoforms: 15
Using threshold 0.2762912046850136, with max acc 0.8627079077910709
Total chosen isoforms: 15
Using threshold 0.2762912046850136, with max acc 0.8627079077910709
Total chosen isoforms: 17
Using threshold 0.3102342179491106, with max acc 0.8660636124890575
Total chosen isoforms: 19
Us

In [29]:
isoform_count_by_subtype

{'PHlike': 8,
 'HYPER': 2,
 'TCF3PBX1': 1,
 'KMT2A': 1,
 'ETV6RUNX1': 4,
 'DUX4IGH': 2,
 'BCRABL1': 10,
 'PAX5': 10,
 'HYPO': 5,
 'iAMP21': 8}

In [31]:
for subtype in acc_by_subtype:
    print(f"{subtype}: {'%.02f' % (acc_by_subtype[subtype] * 100)}%")

open(f"../../preprocessed/{category}/gene_count_by_subtype_{method}_{metric}.json", "w").write(json.dumps(isoform_count_by_subtype))

PHlike: 86.27%
HYPER: 88.28%
TCF3PBX1: 100.00%
KMT2A: 99.67%
ETV6RUNX1: 98.20%
DUX4IGH: 100.00%
BCRABL1: 95.83%
PAX5: 92.69%
HYPO: 94.10%
iAMP21: 87.75%


133

In [32]:
isoform_set = set()
for subtype in importances:
  print(f"===== {subtype} =====")
  for sex in importances[subtype]:
    print(f"===== {sex} =====")
    for isoform in importances[subtype][sex][:isoform_count_by_subtype[subtype]]:
      isoform_set.add(isoform["isoform"])
      print(f"{isoform['isoform']} - {'%.04f' % isoform['coef']}")

open(f"../../preprocessed/{category}/one_vs_all_isoform_set.json", "w").write(json.dumps(list(isoform_set)))

===== BCRABL1 =====
===== Male =====
ENST00000306121 - 0.3482
ENST00000420196 - 0.2609
ENST00000406333 - 0.1703
ENST00000651973 - 0.1703
ENST00000383052 - 0.1101
ENST00000664650 - 0.1059
ENST00000484414 - 0.1001
ENST00000318560 - 0.0531
ENST00000518053 - 0.0495
ENST00000520549 - 0.0485
===== Female =====
ENST00000250784 - 1.3286
ENST00000692644 - 0.3461
ENST00000447520 - 0.2710
ENST00000363484 - 0.2180
ENST00000637996 - 0.2033
ENST00000420196 - 0.1805
ENST00000440408 - 0.1659
ENST00000516768 - 0.1563
ENST00000472056 - 0.1491
ENST00000651177 - 0.1150
===== DUX4IGH =====
===== Male =====
ENST00000265404 - 0.2521
ENST00000685465 - 0.1090
===== Female =====
ENST00000250784 - 0.1681
ENST00000483262 - 0.1638
===== ETV6RUNX1 =====
===== Male =====
ENST00000636884 - 0.3060
ENST00000473821 - 0.2705
ENST00000597477 - 0.2027
ENST00000511142 - 0.1874
===== Female =====
ENST00000572062 - 0.8524
ENST00000511142 - 0.1901
ENST00000641814 - 0.1747
ENST00000682078 - 0.1677
===== PHlike =====
===== Male 

1596

# LaTeX

In [33]:
from latex import generate_one_vs_all_feature_table

table = generate_one_vs_all_feature_table(importances=importances, feature_count_by_subtype=isoform_count_by_subtype, acc_by_subtype=acc_by_subtype, feature_name="isoform", importance_name="coef", importance_scale=100)
print(table)

\begin{tabular}{l c l p{9.5cm}}
\hline
\textbf{Subtype} & \textbf{Max BAcc. (\%)} & \textbf{Sex} & \textbf{Features ({coef} x {100})} \\
\hline
\multirow{2}{*}{BCRABL1} & \multirow{2}{*}{0.96} & Male & ENST00000306121 (34.82), ENST00000420196 (26.09), ENST00000406333 (17.03), ENST00000651973 (17.03), ENST00000383052 (11.01), ENST00000664650 (10.59), ENST00000484414 (10.01), ENST00000318560 (5.31), ENST00000518053 (4.95), ENST00000520549 (4.85) \\
& & Female & ENST00000250784 (132.86), ENST00000692644 (34.61), ENST00000447520 (27.10), ENST00000363484 (21.80), ENST00000637996 (20.33), ENST00000420196 (18.05), ENST00000440408 (16.59), ENST00000516768 (15.63), ENST00000472056 (14.91), ENST00000651177 (11.50) \\
\hline
\multirow{2}{*}{DUX4IGH} & \multirow{2}{*}{1.00} & Male & ENST00000265404 (25.21), ENST00000685465 (10.90) \\
& & Female & ENST00000250784 (16.81), ENST00000483262 (16.38) \\
\hline
\multirow{2}{*}{ETV6RUNX1} & \multirow{2}{*}{0.98} & Male & ENST00000636884 (30.60), ENST00000